# Regression and genuine held-out evaluation

Fixed model specifications; no test-set tuning. These estimate simultaneous ground speed, not future trajectory. Later-day and other-aircraft tests answer different questions. Record train/test dates and group sizes. Do not present results until real-data coverage is adequate.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
with closing(sqlite3.connect(DB)) as db:
    data = pd.read_sql_query("SELECT * FROM observations", db)
    airports = pd.read_sql_query("SELECT * FROM airports", db)
    audit = pd.read_sql_query("SELECT * FROM integration_audit", db)
data["observed_at_utc"] = pd.to_datetime(data.observed_at_utc, utc=True)
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

work = data[data.zone.isin(["near","far"])].copy()
work["near_indicator"] = work.zone.eq("near").astype(int)
work["altitude_x_near"] = work.geo_altitude * work.near_indicator
altitude = ["geo_altitude"]
spatial = ["geo_altitude","distance_km","latitude","longitude","near_indicator","altitude_x_near"]
assert not {"velocity","speed_kmh"}.intersection(spatial)
specs = {
    "Mean baseline": (altitude, lambda: DummyRegressor(strategy="mean")),
    "Altitude linear": (altitude, lambda: make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LinearRegression())),
    "Spatial interaction linear": (spatial, lambda: make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LinearRegression())),
    "Spatial random forest": (spatial, lambda: make_pipeline(SimpleImputer(strategy="median"),RandomForestRegressor(
        n_estimators=150,max_depth=10,min_samples_leaf=10,random_state=42,n_jobs=-1)))
}
dates = sorted(work.date_utc.unique())
if len(dates) < 7:
    raise RuntimeError("At least seven observed UTC dates required; prefer 14 complete collection days.")
cut = max(1,int(len(dates)*.7))
train = work[work.date_utc.isin(dates[:cut])]
test = work[work.date_utc.isin(dates[cut:])]
print("Training dates:", dates[:cut], "Test dates:", dates[cut:])
print("Shared aircraft in temporal split:",len(set(train.icao24) & set(test.icao24)))
display(pd.concat([train.groupby("zone").size().rename("train"),test.groupby("zone").size().rename("test")],axis=1))
if len(train) < 100 or len(test) < 30 or train.zone.nunique()<2 or test.zone.nunique()<2:
    raise RuntimeError("Insufficient observations/group coverage for the planned evaluation; revise collection.")

In [ ]:
def evaluate(train_frame, test_frame, split_name):
    rows, predictions = [], test_frame.copy()
    for name, (features, factory) in specs.items():
        model = factory()
        model.fit(train_frame[features],train_frame.velocity)
        predicted = model.predict(test_frame[features])
        predictions[name] = predicted
        for group in ["all","near","far"]:
            mask = np.ones(len(test_frame),dtype=bool) if group=="all" else test_frame.zone.eq(group).to_numpy()
            if mask.sum()<2:
                continue
            actual = test_frame.velocity.to_numpy()[mask]
            pred = predicted[mask]
            rows.append({"split":split_name,"model":name,"group":group,"n":int(mask.sum()),
                "aircraft":test_frame.loc[mask,"icao24"].nunique(),
                "MAE_m_s":mean_absolute_error(actual,pred),
                "RMSE_m_s":np.sqrt(mean_squared_error(actual,pred)),"R_squared":r2_score(actual,pred)})
    return pd.DataFrame(rows), predictions

metrics, predictions = evaluate(train,test,"later_days")
display(metrics)
metrics.to_csv(OUT / "tables/temporal_metrics.csv",index=False)
predictions.to_csv(OUT / "tables/temporal_predictions.csv",index=False)
# Diagnostics for a prespecified model, not chosen by inspecting final test scores.
predictions["residual_m_s"] = predictions.velocity - predictions["Spatial interaction linear"]
sns.scatterplot(data=predictions.sample(min(3000,len(predictions)),random_state=42),
    x="geo_altitude",y="residual_m_s",hue="zone",alpha=.4)
plt.axhline(0,color="black",lw=1);plt.xlabel("Geometric altitude (m)");plt.ylabel("Observed minus predicted (m/s)")
plt.tight_layout();plt.savefig(OUT / "figures/residuals_by_altitude.png",dpi=160);plt.show()
display(predictions.groupby(["airport_ident","zone"]).residual_m_s.agg(["count","median","std"]))

In [ ]:
if work.icao24.nunique()<30:
    raise RuntimeError("Too few distinct aircraft for the planned secondary transfer evaluation.")
gss = GroupShuffleSplit(n_splits=1,test_size=.3,random_state=42)
train_idx,test_idx = next(gss.split(work,groups=work.icao24))
group_train,group_test = work.iloc[train_idx],work.iloc[test_idx]
assert set(group_train.icao24).isdisjoint(set(group_test.icao24))
if group_train.zone.nunique()<2 or group_test.zone.nunique()<2:
    raise RuntimeError("Held-out-aircraft split lacks both proximity groups; collect more coverage.")
other_metrics, _ = evaluate(group_train,group_test,"other_aircraft_same_period")
display(other_metrics)
other_metrics.to_csv(OUT / "tables/aircraft_transfer_metrics.csv",index=False)

## Interpretation and planned extensions

Explain whether spatial features improve each evaluation and whether group error differs. Compare overlapping altitude bands and avoid causal language. Add an aircraft-balanced metric and cluster-bootstrap uncertainty if time permits. If tuning is introduced, use validation dates inside training only; keep this final test untouched. A group split across the full period tests new aircraft, not future days.

## Optional fixed comparison: A altitude, B spatial, C spatial + category

Run only after the original multi-day coverage guard passes. All three models use the identical existing training/test observations, including unmatched metadata as unknown/other. We reuse the exact temporal and aircraft splits; no new split or complete-case subset is selected. C learns category encoding on training data only and ignores unseen levels. Static metadata can be outdated; held-out-aircraft testing helps assess whether improvements transfer beyond aircraft mix. This is an extension, not a replacement for the baseline, random forest or original diagnostics.

In [ ]:
from contextlib import closing
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.model_comparison import compare_models
with closing(sqlite3.connect(DB)) as db:
    has_enrichment = db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name='observations_enriched'").fetchone()[0]
    if has_enrichment:
        category_keys = pd.read_sql_query("SELECT icao24,time_position,aircraft_category FROM observations_enriched", db)
        metadata_available = bool(pd.read_sql_query("SELECT source_available FROM metadata_integration_audit", db).iloc[0,0])
    else:
        metadata_available = False
if metadata_available:
    def attach_category(frame):
        enriched = frame.merge(category_keys, on=["icao24", "time_position"], how="left", validate="one_to_one")
        assert len(enriched) == len(frame)
        assert enriched[["icao24", "time_position"]].equals(frame[["icao24", "time_position"]].reset_index(drop=True))
        enriched["aircraft_category"] = enriched.aircraft_category.fillna("unknown/other")
        return enriched
    abc_temporal = compare_models(attach_category(train), attach_category(test), "later_days")
    abc_aircraft = compare_models(attach_category(group_train), attach_category(group_test), "other_aircraft_same_period")
    abc = pd.concat([abc_temporal, abc_aircraft], ignore_index=True)
    display(abc)
    abc.to_csv(OUT / "tables/optional_metadata_comparison.csv", index=False)
else:
    print("Optional metadata unavailable: A/B/C extension skipped; original analysis remains available.")
